# Notebook 01 — Read & Join the Tables

## Objective

Read the Olist tables from PostgreSQL, aggregate one-to-many tables,
and build an order-level ML table.

## Output

`artifacts/notebook_01/ml_table.parquet`

## Grain

One row = one order

In [ ]:
pip install pandas numpy sqlalchemy psycopg2-binary pyarrow

In [7]:
import pandas as pd
import numpy as np

from sqlalchemy import create_engine

DB_USER = "olist_user"
DB_PASSWORD = "olist_password"
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "olist_db"

DATABASE_URL = (
    f"postgresql+psycopg2://"
    f"{DB_USER}:{DB_PASSWORD}@"
    f"{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)

In [8]:
with engine.connect() as connection:
    print("Database connection successful!")
    print(f"Connected to: {DB_NAME} on {DB_HOST}:{DB_PORT}")

Database connection successful!
Connected to: olist_db on localhost:5432


In [9]:
tables = [
    "customers",
    "geolocation",
    "orders",
    "order_items",
    "order_payments",
    "order_reviews",
    "products",
    "sellers",
    "product_category_name_translation",
]
data = {}

for table in tables:
    data[table] = pd.read_sql_table(table, engine)
    print(f"{table:45} {data[table].shape}")

customers                                     (99441, 5)
geolocation                                   (1000163, 5)
orders                                        (99441, 8)
order_items                                   (112650, 7)
order_payments                                (103886, 5)
order_reviews                                 (99224, 7)
products                                      (32951, 9)
sellers                                       (3095, 4)
product_category_name_translation             (71, 2)


In [10]:
for table_name, df in data.items():
    print("=" * 80)
    print(f"TABLE: {table_name}")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")
    print("\nColumns:")
    print(df.columns.tolist())
    print()

TABLE: customers
Rows: 99,441
Columns: 5

Columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

TABLE: geolocation
Rows: 1,000,163
Columns: 5

Columns:
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

TABLE: orders
Rows: 99,441
Columns: 8

Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

TABLE: order_items
Rows: 112,650
Columns: 7

Columns:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

TABLE: order_payments
Rows: 103,886
Columns: 5

Columns:
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

TABLE: order_reviews
Rows: 99,224
Columns: 7

Columns:
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review

In [11]:
orders = data["orders"]

print("Orders:", len(orders))
print("Unique order_id:", orders["order_id"].nunique())
print("Duplicated order_id:", orders["order_id"].duplicated().sum())

Orders: 99441
Unique order_id: 99441
Duplicated order_id: 0


In [12]:
order_items = data["order_items"]

print("Rows:", len(order_items))
print("Unique order_id:", order_items["order_id"].nunique())
print("Unique (order_id, order_item_id):",
      order_items[["order_id", "order_item_id"]].drop_duplicates().shape[0])

Rows: 112650
Unique order_id: 98666
Unique (order_id, order_item_id): 112650


In [13]:
payments = data["order_payments"]

print("Rows:", len(payments))
print("Unique order_id:", payments["order_id"].nunique())
print(
    "Unique (order_id, payment_sequential):",
    payments[["order_id", "payment_sequential"]].drop_duplicates().shape[0]
)

Rows: 103886
Unique order_id: 99440
Unique (order_id, payment_sequential): 103886


In [14]:
items_agg = (
    order_items
    .groupby("order_id")
    .agg(
        item_count=("order_item_id", "count"),
        total_price=("price", "sum"),
        total_freight_value=("freight_value", "sum"),
        unique_products=("product_id", "nunique"),
        unique_sellers=("seller_id", "nunique"),
    )
    .reset_index()
)

items_agg.head()

,order_id,item_count,total_price,total_freight_value,unique_products,unique_sellers
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,1,1
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,1,1
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,1,1
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,1,1


In [15]:
items_agg["order_id"].is_unique

True

In [16]:
payments_agg = (
    payments
    .groupby("order_id")
    .agg(
        payment_count=("payment_sequential", "count"),
        total_payment_value=("payment_value", "sum"),
        payment_types_count=("payment_type", "nunique"),
    )
    .reset_index()
)

payments_agg.head()

,order_id,payment_count,total_payment_value,payment_types_count
0,00010242fe8c5a6d1ba2dd792cb16214,1,72.19,1
1,00018f77f2f0320c557190d7a144bdd3,1,259.83,1
2,000229ec398224ef6ca0657da4fc703e,1,216.87,1
3,00024acbcdf0a6daa1e931b038114c75,1,25.78,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,218.04,1


In [17]:
payments_agg["order_id"].is_unique

True

## Build the ML Table

In [18]:
ml_table = orders.copy()

In [19]:
customer_features = data["customers"][
    [
        "customer_id",
        "customer_city",
        "customer_state",
        "customer_zip_code_prefix",
    ]
]

ml_table = ml_table.merge(
    customer_features,
    on="customer_id",
    how="left",
    validate="one_to_one",
)

In [20]:
ml_table = ml_table.merge(
    items_agg,
    on="order_id",
    how="left",
    validate="one_to_one",
)

In [21]:
ml_table = ml_table.merge(
    payments_agg,
    on="order_id",
    how="left",
    validate="one_to_one",
)

In [22]:
print("Rows:", len(ml_table))
print("Unique order_id:", ml_table["order_id"].nunique())
print("order_id is unique:", ml_table["order_id"].is_unique)

Rows: 99441
Unique order_id: 99441
order_id is unique: True


In [23]:
print("Orders without item aggregation:")
print(ml_table["item_count"].isna().sum())

print("Orders without payment aggregation:")
print(ml_table["payment_count"].isna().sum())

Orders without item aggregation:
775
Orders without payment aggregation:
1


## Save Artifact

In [24]:
from pathlib import Path

artifact_dir = Path("../artifacts/notebook_01")
artifact_dir.mkdir(parents=True, exist_ok=True)

In [25]:
output_path = artifact_dir / "ml_table.parquet"

ml_table.to_parquet(output_path, index=False)

print(f"Saved ML table to: {output_path}")

Saved ML table to: ..\artifacts\notebook_01\ml_table.parquet


In [26]:
saved_ml_table = pd.read_parquet(output_path)

print("Saved table shape:", saved_ml_table.shape)
print("Unique orders:", saved_ml_table["order_id"].nunique())
print("Order IDs unique:", saved_ml_table["order_id"].is_unique)

Saved table shape: (99441, 19)
Unique orders: 99441
Order IDs unique: True


In [27]:
ml_table.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'customer_city',
 'customer_state',
 'customer_zip_code_prefix',
 'item_count',
 'total_price',
 'total_freight_value',
 'unique_products',
 'unique_sellers',
 'payment_count',
 'total_payment_value',
 'payment_types_count']

## Validation

Verify that the final table preserves the one-row-per-order grain.

Checks include:
- Row count
- Unique `order_id`
- Orders without items
- Orders without payments
- Missing values after joins

In [28]:
print("ML table shape:", ml_table.shape)
print("Unique orders:", ml_table["order_id"].nunique())
print("order_id is unique:", ml_table["order_id"].is_unique)

print("\nOrders without items:",
      ml_table["item_count"].isna().sum())

print("Orders without payments:",
      ml_table["payment_count"].isna().sum())

ML table shape: (99441, 19)
Unique orders: 99441
order_id is unique: True

Orders without items: 775
Orders without payments: 1


In [29]:
assert len(ml_table) == len(orders)
assert ml_table["order_id"].nunique() == len(orders)
assert ml_table["order_id"].is_unique

print("Validation passed: one row per order.")

Validation passed: one row per order.


In [30]:
assert items_agg["order_id"].is_unique
assert payments_agg["order_id"].is_unique

print("Aggregation validation passed.")

Aggregation validation passed.


## Summary

- Loaded and inspected all nine source tables.
- Aggregated one-to-many tables before joining.
- Built an order-level ML table.
- Verified one row per order.
- Saved the final Parquet artifact.

### Result

- Rows: 99,441
- Unique orders: 99,441
- `order_id` unique: True
- Orders without items: 775
- Orders without payments: 1

### Output

`artifacts/notebook_01/ml_table.parquet`